# Imports de librerias

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import StringType
from pyspark.sql.functions import trim, col, lower, count, countDistinct, when, length, translate, avg, first, lpad

# Lectura de la Tabla Bronce olist_customers

In [0]:
# creo el df con la tabla bronce de olist_customers
df = spark.table("`catalog_brazilian-e-commerce`.bronze.olist_geolocation_dataset")

In [0]:
df.display()

# Transformaciones

In [0]:
# se limpian los espacios y se deja en minuscula geolocation_city
df = df.withColumn(
    "geolocation_city",
    lower(trim(col("geolocation_city")))
)

In [0]:
# se quitan los acentos de geolocation_city
df = df.withColumn(
    "geolocation_city",
    translate(col("geolocation_city"),
              "áàãâäéèêëíìîïóòõôöúùûüç",
              "aaaaaeeeeiiiiooooouuuuc")
)

In [0]:
# identifico los zip_code duplicados (sucede porque cambian muchas veces las coordenadas de latitud y longitud)
df.groupBy("geolocation_zip_code_prefix") \
  .count() \
  .orderBy("count", ascending=False) \
  .show()

In [0]:

# asegurar 1 sola ciudad por ZIP
zip_city_issues = df.groupBy("geolocation_zip_code_prefix") \
    .agg(countDistinct("geolocation_city").alias("cities")) \
    .filter("cities > 1")
zip_city_issues.show()


In [0]:
# agrupo por geolocation_zip_code_prefix y se promedian lat y lng, se toma el primer valor de city y state. Adenas se les cambi
df = df.groupBy("geolocation_zip_code_prefix") \
    .agg(
        avg("geolocation_lat").alias("geolocation_lat"),
        avg("geolocation_lng").alias("geolocation_lng"),
        first("geolocation_city", ignorenulls=True).alias("geolocation_city"),
        first("geolocation_state", ignorenulls=True).alias("geolocation_state")
    )

In [0]:
df.display()

In [0]:
# chequeo que haya quedado 1 solo zip_code
df.groupBy("geolocation_zip_code_prefix") \
  .count() \
  .orderBy("count", ascending=False) \
  .show()

In [0]:
df.select("geolocation_zip_code_prefix").distinct().count()
df.count()

In [0]:
# chequeo que no haya ningun valor nulo en las columnas
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

In [0]:

# se verifica que las coordenadas sean validas
df.filter(
    (col("geolocation_lat") > 90) | (col("geolocation_lat") < -90) |
    (col("geolocation_lng") > 180) | (col("geolocation_lng") < -180)
).show()

In [0]:
# el geolocation_zip_code_prefix debe tener 5 digitos
df = df.withColumn(
    "geolocation_zip_code_prefix",
    lpad(col("geolocation_zip_code_prefix").cast("string"), 5, "0")
)

# Crear la tabla Silver de olist_geolocation

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("`catalog_brazilian-e-commerce`.silver.olist_geolocation")

In [0]:
%sql
select * from `catalog_brazilian-e-commerce`.silver.olist_geolocation